# Telco churn: exploratory analysis

This notebook gives a reproducible overview of the bundled IBM sample data. Training and artifact generation live in `model/train.py` so the notebook and app share the same pipeline.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path('data/telco_churn.csv'), Path('../data/telco_churn.csv')]
DATA = next(path for path in candidates if path.exists())
df = pd.read_csv(DATA)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df.head()

In [ ]:
print('Rows, columns:', df.shape)
print('Missing values:', df.isna().sum().sort_values(ascending=False).head())
print('Churn rate:', (df['Churn'] == 'Yes').mean())
df.describe(include='all').T.head(15)

In [ ]:
contract_churn = df.assign(churned=df['Churn'].eq('Yes')).groupby('Contract', observed=True)['churned'].mean().sort_values(ascending=False)
contract_churn.plot(kind='bar', ylabel='Churn rate', title='Observed churn by contract')
plt.tight_layout()

In [ ]:
df.assign(churned=df['Churn'].eq('Yes')).groupby('tenure', observed=True)['churned'].mean().plot(ylabel='Observed churn rate', title='Observed churn by tenure month')
plt.tight_layout()

## Train and evaluate
Run `python -m model.train` from the repository root to fit the calibrated XGBoost pipeline, compare the logistic regression baseline, choose the retention threshold from validation data, and save test metrics and calibration output. The threshold and financial values are estimates under the assumptions documented in `src/config.py`.